# Project 03 (final): spam filter with naive Bayes — on real data

**Goal:** you build a complete spam classifier **entirely from scratch** — with
nothing but Bayes' rule, counting and logarithms — and measure it on real data:
5,574 real text messages (about 13 % of them spam) from the *SMS Spam Collection* (UCI).
At the end you compare your model with the scikit-learn implementation.

**Preparation:** once in the terminal (from the folder `03-final`, venv active):

```
python datasets/download_data.py
```

**Reference to the script:** section 2.4 (Bayes' rule, naive Bayes) and 2.5 (supervised learning).

## 1. Load the data and look at it

Rule number one with real data: **look first, model second**.

**Task:** load `datasets/SMSSpamCollection` (tab-separated, no header row, columns `label` and `text`; `pd.read_csv(..., sep="\t", header=None, names=["label","text"], quoting=3)` — `quoting=3` prevents `"` from being interpreted as a quote character). Then get an overview: class distribution (`value_counts`), spam share (about 13 %), and compare the text length of ham vs. spam (`str.len`, `groupby`).

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Important — imbalanced classes:** only about 13 % spam. A "classifier" that
stubbornly says *ham* would already have about 87 % accuracy! So accuracy alone
is not an adequate performance measure here — we check this later with precision
and recall.

## 2. Training and test set

We evaluate the model only on messages it has **never seen** during training
(script 2.5: generalisation!). `stratify` ensures that the spam share is the same
in both subsets; the fixed `random_state` makes everything reproducible.

**Task:** split with `train_test_split` into 80 % training / 20 % test (`test_size=0.2`, `random_state=42`, `stratify=labels`) and check that the spam share is about equal in both subsets.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 3. From text to words: tokenisation

Naive Bayes works with word probabilities — so we have to break messages into
words. We keep it deliberately simple: lowercase, then extract all sequences of
letters and digits.

**Task:** implement `tokenize(text)` yourself.

**Hint:** use the regex `[a-z0-9']+` and lowercase the text beforehand (`.lower()`). **Self-check:** `tokenize("WINNER!! Claim your £900 prize now!")` must give `['winner','claim','your','900','prize','now']`.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 4. Naive Bayes from scratch

As a reminder (script 2.4): for a message with words $w_1, \dots, w_n$ we compare

$$P(\text{spam} \mid w_1..w_n) \propto P(\text{spam}) \prod_i P(w_i \mid \text{spam})
\qquad \text{vs.} \qquad
P(\text{ham} \mid w_1..w_n) \propto P(\text{ham}) \prod_i P(w_i \mid \text{ham})$$

Two practical tricks, both of which you implement yourself:

1. **Log probabilities:** the product of hundreds of small numbers would collapse
   numerically to 0 (*underflow*). We therefore compute with sums of logarithms:
   $\log P(c) + \sum_i \log P(w_i \mid c)$ — the comparison stays the same,
   because the logarithm is monotonic.
2. **Laplace smoothing:** a word that never occurred in spam during training would
   have $P(w \mid \text{spam}) = 0$ — one single such word would "acquit" every
   spam message ($\log 0 = -\infty$). We therefore pretend we had seen every known
   word in every class **one extra time** ($\alpha = 1$):

$$P(w \mid c) = \frac{\text{count}(w, c) + 1}{\text{total words in } c + |V|}$$

where $|V|$ is the size of the vocabulary (all known words).

**Task:** build `train(texts, labels)` (counts the words per class, the vocabulary and the priors), `log_word_prob(model, word, cls)` (the Laplace formula above), `log_posterior(model, text, cls)` (prior + sum of the log word probabilities over the known words) and `classify(model, text)`. **Self-check:** `"URGENT! You have won a free prize, call now!"` → `spam`, `"Ok, see you at the station at 6"` → `ham`.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 5. How good is the filter really?

Now the acid test on the held-out test data. Besides accuracy we look at the
**confusion matrix** and two figures:

- **Precision** (spam): when the filter says "spam" — how often is that right?
  *(Important: false positives = genuine messages in the spam folder = very annoying!)*
- **Recall** (spam): how much of the actual spam does the filter catch?

**Task:** compute `accuracy_score`, `confusion_matrix` and `classification_report` on the test data. **Expectation:** accuracy about 0.98 — clearly above the "always ham" baseline (about 0.87). Pay particular attention to precision and recall of the spam class.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 6. What did the model learn?

A big advantage of naive Bayes: you can **look inside**. Which words argue most
strongly for spam? We rank by the log ratio
$\log \frac{P(w \mid \text{spam})}{P(w \mid \text{ham})}$ (only words occurring at least 5 times).

**Hint:** rank the words by the log ratio $\log P(w\mid\text{spam}) - \log P(w\mid\text{ham})$ and consider only words that occur at least 5 times in total.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 7. Comparison with scikit-learn

To finish, the same model with the standard tools of practice:
`CountVectorizer` (counts words) + `MultinomialNB` (exactly our algorithm).
If your by-hand version is good, the two are close together.

**Hint:** `CountVectorizer(token_pattern=r"[a-z0-9']+", lowercase=True)` + `MultinomialNB(alpha=1.0)`. Your by-hand accuracy should be very close to the scikit-learn accuracy.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


*(Small deviations are normal: scikit-learn counts words that occur several times
in a message with their multiplicity, while details such as the handling of unknown
words are solved slightly differently in our version.)*

## Done — what you can do now

- load a real data set, explore it and split it cleanly into train/test
- translate Bayes' rule into a working classifier
  (including the two practical tricks: log space and Laplace smoothing)
- evaluate a model with the *right* metrics (precision/recall instead of accuracy alone)
- benchmark your by-hand model against an industry implementation

**Bonus tasks** (optional, no reference solution):
1. The filter puts some genuine messages into the spam folder (false positives). Look
   at those messages (`test_texts[(predictions_series == "spam") & (test_labels == "ham")]`) — why does the model stumble?
2. Experiment with the smoothing: what happens at $\alpha = 0.01$ or $\alpha = 10$?
3. Remove words that occur only once from the vocabulary. Does the model get better or
   worse — and why could either happen?